Programme: DATA5000_Simple_RAG.ipynb
Author: Sanjeev Naguleswaran
Date: December 2024

Building a Vector Database with HuggingFace Embeddings and Querying

This notebook will walk you through creating a vector database of documents using HuggingFace embeddings and then querying it using GPT-Neo. By the end, you’ll be able to store documents as embeddings in a vector database and retrieve relevant information from them.

In [ ]:
# This could be run from your Python environment once
#import os

# Install all required packages from a Python script

%pip install langchain chromadb tokenizers sentence-transformers langchain-huggingface
%pip install -U langchain-community

zsh:1: unmatched '


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Note: you may need to restart the kernel to use updated packages.


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Note: you may need to restart the kernel to use updated packages.


This installs:
	•	langchain: The framework for building AI/LLM-based applications.
	•	chromadb: Vector database for embeddings.
	•	tokenizers: Used for efficient tokenization of input text for embedding models.

##Import Required Libraries##

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_core.documents import Document

##Create Sample Documents##
We will create a list of simple text documents. Each document will be converted into a vector and stored in ChromaDB.

In [ ]:
documents = [
    "Machine learning enables machines to learn from data without being explicitly programmed.",
    "Deep learning is a subset of machine learning that uses artificial neural networks.",
    "ChromaDB is a vector database for storing and querying embeddings with support for metadata and persistence.",
    "HuggingFace transformers are state-of-the-art models for various natural language processing (NLP) tasks.",
    "Natural language processing (NLP) is a field of AI that enables computers to understand human language.",
    "Python is one of the most popular programming languages for machine learning, AI, and data science.",
    "LangChain provides a framework to chain together multiple AI models and tools to create pipelines for large-scale applications."
]

##Initialize the Embedding Model##

We will use HuggingFace Embeddings to convert documents into dense vectors that can be stored in ChromaDB.

In [ ]:


# Initialize the HuggingFace embedding model
#This model converts sentences into 384-dimensional dense vectors.
embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

/var/folders/13/1spfd0g145lgtmt62rrlrp440000gn/T/ipykernel_36015/80538039.py:5: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
/opt/anaconda3/envs/LLM/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


#Create a ChromaDB Vector Store##

We will create a Chroma vector store to persist embeddings, metadata, and document text. The vector store will be saved to disk in the ./chroma_storage/ directory.


In [ ]:
import tempfile


# Create a temporary directory to store the ChromaDB
temp_dir = tempfile.mkdtemp()


# Create the Chroma vector store (persist to disk)
chroma_db = Chroma(
    collection_name="example_docs",
    embedding_function=embedding_model,
    persist_directory=temp_dir
)
# Convert plain text into Document objects
document_objects = [Document(page_content=doc) for doc in documents]

# Add documents to ChromaDB
chroma_db.add_documents(document_objects)
chroma_db.persist()

print(f"ChromaDB setup complete. Database has been saved in {temp_dir}")

/var/folders/13/1spfd0g145lgtmt62rrlrp440000gn/T/ipykernel_36015/2487054453.py:10: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-chroma package and should be used instead. To use it run `pip install -U :class:`~langchain-chroma` and import as `from :class:`~langchain_chroma import Chroma``.
  chroma_db = Chroma(


ChromaDB setup complete. Database has been saved in /var/folders/13/1spfd0g145lgtmt62rrlrp440000gn/T/tmp5p1ni_nw


/var/folders/13/1spfd0g145lgtmt62rrlrp440000gn/T/ipykernel_36015/2487054453.py:20: LangChainDeprecationWarning: Since Chroma 0.4.x the manual persistence method is no longer supported as docs are automatically persisted.
  chroma_db.persist()


##Query the ChromaDB##

After creating and persisting the vector store, you can query it to find similar documents.

In [ ]:
# Reload the ChromaDB vector store from the temp directory
chroma_db = Chroma(
    collection_name="example_docs",
    embedding_function=embedding_model,
    persist_directory=temp_dir
)

# Query the ChromaDB vector store
query = "What is machine learning?"
results = chroma_db.similarity_search(query, k=3)  # Get the top 3 most similar documents

# Display the results
print(f"\n🔍 Query: {query}\n")
for i, result in enumerate(results):
    print(f"Result {i+1}:")
    print(f"Document: {result.page_content}\n")


🔍 Query: What is machine learning?

Result 1:
Document: Machine learning enables machines to learn from data without being explicitly programmed.

Result 2:
Document: Deep learning is a subset of machine learning that uses artificial neural networks.

Result 3:
Document: Natural language processing (NLP) is a field of AI that enables computers to understand human language.



##Add Documents with Metadata##

You can also add metadata to your documents. This metadata can later be used to filter search results.

In [ ]:
# Documents with metadata
documents_with_metadata = [
    {"content": "Machine learning is the process of enabling computers to learn from data.",
     "metadata": {"source": "AI Journal", "category": "AI"}},
    {"content": "Deep learning relies on deep neural networks to perform complex tasks.",
     "metadata": {"source": "AI Research Paper", "category": "AI"}},
    {"content": "ChromaDB provides metadata support, which is useful for document search.",
     "metadata": {"source": "Database Journal", "category": "Database"}},
]

# Convert plain text into Document objects with metadata
document_objects_with_metadata = [
    Document(page_content=doc["content"], metadata=doc["metadata"]) for doc in documents_with_metadata
]

# Add the documents with metadata to ChromaDB
chroma_db.add_documents(document_objects_with_metadata)
chroma_db.persist()

print("✅ Added documents with metadata.")

✅ Added documents with metadata.


Explanation of Key Parts:
	•	metadata: Each document can include additional attributes like source, author, or category.

##Query with Metadata Filters##

Now, let’s query ChromaDB but only retrieve documents in the AI category.

Explanation of Key Parts:
	•	filter={"category": "AI"}: This filters results to only return documents where category = 'AI'.

In [ ]:
# Query the vector database using a filter on metadata
query = "Explain deep learning."
results = chroma_db.similarity_search(query, k=3, filter={"category": "AI"})

# Display the results
print(f"\n🔍 Query with Filter (Category = 'AI') for: {query}\n")
for i, result in enumerate(results):
    print(f"Result {i+1}:")
    print(f"Document: {result.page_content}")
    print(f"Metadata: {result.metadata}\n")


🔍 Query with Filter (Category = 'AI') for: Explain deep learning.

Result 1:
Document: Deep learning relies on deep neural networks to perform complex tasks.
Metadata: {'category': 'AI', 'source': 'AI Research Paper'}

Result 2:
Document: Machine learning is the process of enabling computers to learn from data.
Metadata: {'category': 'AI', 'source': 'AI Journal'}



In [12]:
# Query the vector database again using a filter on metadata
query = "What is ChromaDB all about?"
results = chroma_db.similarity_search(query, k=3, filter={"category": "Database"})

# Display the results
print(f"\n🔍 Query with Filter (Category = 'Database') for: {query}\n")
for i, result in enumerate(results):
    print(f"Result {i+1}:")
    print(f"Document: {result.page_content}")
    print(f"Metadata: {result.metadata}\n")